In [255]:
def get_the_right_cable(switch,absolute_sw_to_dev, devspeed):
    
    cables = Cables[switch]

   
    higher_lengths =(
     sorted([item for item in cables if isinstance(item.get('length'), 
                                    (int, float)) and item.get('length') > absolute_sw_to_dev
                                    and item.get('server_port_speed') == devspeed], key=lambda x: x['length'])
    )
   
    if higher_lengths:
        
        return higher_lengths[0]
    else:
        return {}

In [256]:
def calculate_cable_lengths(rack):
    
    lans = dict()
    devices = dict()
    switches = dict()
    for dev in rack:
        dev_id = dev.split('_')[-1]
        device = '_'.join(dev.split('_')[:-1])
        if 'LAN' in device:
            lanname, switch = device.split('__')
            lans[dev] = dict()
            if '_spine' in lanname:
                device_specs = [x for x in Spines[lanname]['switch'] if x['model'] == switch ][0]
            else:   
                device_specs = [x for x in LANs[lanname]['switch'] if x['model'] == switch ][0]
            lans[dev] = copy.deepcopy(device_specs)
            lans[dev]['position'] = rack[dev]
            lans[dev]['id'] = dev_id
            lans[dev]['half_for_split'] = 0
            lans[dev]['full_no_split'] = 0
            lans[dev]['cables'] = dict()
            lans[dev]['minimum_cable_total_length'] = 0
            lans[dev]['actual_cable_total_length'] = 0
            if device not in switches:
                switches[device] = 0
            switches[device] +=1
        else:
            devices[dev] = dict()
            devices[dev]['position'] = rack[dev]
            devices[dev]['id'] = dev_id
          
    for lan in lans:
        lanname = lan.split('__')[0]
        if '_spine' in lanname:
            continue
        switch = lans[lan]['model']
        cable_to_ceil = 0
        for dev in rack:
            dev_id = dev.split('_')[-1]
            device = '_'.join(dev.split('_')[:-1])
            if 'LAN' not in device :
                if colors_info[device][lanname]['count'] > 0:
                    no_ports = colors_info[device][lanname]['count']
                    speed_ratio =  colors_info[device][lanname]['speed'] / lans[lan]['speed']
                    device_position = rack[dev]
                    # absolute cable lengeth in meters:
                    absolute_sw_to_dev = ((lans[lan]['position'] - device_position) + 2*device_to_rackside)*unit_to_cm/100
                    cable = get_the_right_cable(switch,absolute_sw_to_dev,colors_info[device][lanname]['speed'])
                    no_of_cables = colors_info[device][lanname]['count'] * speed_ratio
                    devices[dev][lan] = dict()
                    devices[dev][lan]['cable_count'] = colors_info[device][lanname]['count']
                    devices[dev][lan]['speed'] = colors_info[device][lanname]['speed']
                    devices[dev][lan]['to_external_rack'] = -1
                    if speed_ratio < 1:
                        lans[lan]['half_for_split'] +=1
                        devices[dev][lan]['half_for_split'] = 1
                        cable_to_ceil = cable['model']
                    else:
                        lans[lan]['full_no_split'] += 1
                        devices[dev][lan]['full_no_split'] = 1
                    devices[dev][lan]['cables_count'] = no_of_cables
                    devices[dev][lan]['cable_type'] = cable
                    
                    if cable['model'] not in lans[lan]['cables']:
                        lans[lan]['cables'][cable['model']] = 0                    
                    lans[lan]['cables'][cable['model']] += no_of_cables
                    lans[lan]['minimum_cable_total_length'] += absolute_sw_to_dev * no_of_cables
                    lans[lan]['actual_cable_total_length'] += cable['length'] * no_of_cables
        if cable_to_ceil:
            lans[lan]['cables'][cable_to_ceil] = ceil(lans[lan]['cables'][cable_to_ceil])
    
    return (devices, lans, switches)

rack = {'LAN_1__IB+400_1': 41, 'FrontEnd_nodes_2': 40, 'GPU_nodes_3': 32, 'compute_nodes_4': 30, 'compute_nodes_5': 28, 'compute_nodes_6': 26, 'compute_nodes_7': 24, 'compute_nodes_8': 22}
lans, devices  = calculate_cable_lengths(rack, LANs, Cables, colors_info)
print('devices',devices)
print('lans',lans)

In [257]:
from operator import itemgetter
def sort_device_before_placement(all_items):
    def sort_key(item):
        type_getter = itemgetter('type')
        weight_getter = itemgetter('weight')
        item_type = type_getter(item)
        return (0, item_type) if item_type.startswith('LAN') else (1, -weight_getter(item))

    sorted_list = sorted(all_items, key=sort_key)
    
    return sorted_list

all_servers = [{'model': 'IB+400', 'ports': 64, 'speed': 400, 'height': 2, 'wattage': 2000, 'weight': 20, 'type': 'LAN_3__IB+400'}, {'count': 20, 'wattage': 11000, 'height': 8, 'weight': 15, 'LAN_1': {'count':0, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 8, 'speed': 400}, 'type': 'GPU_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}]

sorted_servers = sort_device_before_placement(all_servers)
print(sorted_servers)

In [258]:
def old_sort_device_before_placement(item):
    if 'LAN' in item['type']:
        return (0, item['type'])  # Prioritize LAN, then sort alphabetically by type
    else:
        return (1, item['type'])  # Other types come later, maintain original order



In [259]:
10//4

2

In [260]:
import numpy as np
from collections import Counter
import functools
import time, copy
from functools import cache, lru_cache, wraps

global_rack_signature = dict()


def is_rack_stable(servers):
    """
    Checks if the rack configuration is likely stable based on the combined
    vertical center of gravity.
    """
    total_weight = rack_weight_kg + sum(s[1] for s in servers)
    if total_weight == 0:
        return True

    combined_vertical_cg = (rack_weight_kg * rack_cg_height_mm +
                             sum(s[1] * s[0] for s in servers)) / total_weight

    stability_threshold_fraction = 0.6  # Adjust as needed
    return combined_vertical_cg <= rack_height_mm * stability_threshold_fraction



import collections
from functools import wraps

def to_hashable(obj):
    """Convert objects to hashable forms while preserving structure."""
    if isinstance(obj, Counter):
        # For Counter, convert to tuple of items
        return ('__counter__', tuple(sorted(obj.items())))
    elif isinstance(obj, dict):
        # For dicts, convert to tuple of sorted items
        return ('__dict__', tuple(sorted((k, to_hashable(v)) for k, v in obj.items())))
    elif isinstance(obj, list):
        return ('__list__', tuple(to_hashable(item) for item in obj))
    elif isinstance(obj, tuple):
        return ('__tuple__', tuple(to_hashable(item) for item in obj)) 
    elif isinstance(obj, set):
        return ('__set__', frozenset(to_hashable(item) for item in obj))
    return obj

def from_hashable(obj):
    """Convert back from hashable form to original objects."""
    if isinstance(obj, tuple) and len(obj) == 2:
        type_tag, value = obj
        if type_tag == '__counter__':
            # Rebuild Counter from items
            return Counter(dict(value))
        elif type_tag == '__dict__':
            # Rebuild dict
            return {k: from_hashable(v) for k, v in value}
        elif type_tag == '__list__':
            # Rebuild list
            return [from_hashable(item) for item in value]
        elif type_tag == '__tuple__':
            # Keep as tuple but convert contents
            return tuple(from_hashable(item) for item in value)
        elif type_tag == '__set__':
            # Rebuild set
            return {from_hashable(item) for item in value}
    
    # If it's a regular tuple (not tagged), process its elements
    if isinstance(obj, tuple):
        return tuple(from_hashable(item) for item in obj)
    
    return obj

def hashable_cache(func):
    """
    Decorator that makes function arguments hashable for caching,
    then converts them back to their original types when calling the function.
    """
    @functools.lru_cache(maxsize=100)
    def cached_wrapper(*hashable_args, **hashable_kwargs):
        # Convert the hashable arguments back to their original types
        restored_args = tuple(from_hashable(arg) for arg in hashable_args)
        restored_kwargs = {k: from_hashable(v) for k, v in hashable_kwargs.items()}
        
        # Call the original function with restored arguments
        return func(*restored_args, **restored_kwargs)
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        # Convert arguments to hashable versions
        hashable_args = tuple(to_hashable(arg) for arg in args)
        hashable_kwargs = {k: to_hashable(v) for k, v in kwargs.items()}
        
        # Call the cached version
        return cached_wrapper(*hashable_args, **hashable_kwargs)
    
    # Add cache control methods
    wrapper.cache_clear = cached_wrapper.cache_clear
    wrapper.cache_info = cached_wrapper.cache_info
    
    return wrapper

# Example usage - rename this to @hashable_args if that's what your code expects
def hashable_args(func):
    return hashable_cache(func)

# Test function to demonstrate usage
@functools.lru_cache(maxsize=100)
def find_stable_positions_greedy_complex( servers_to_place_tuple, prioritize_top=True):
    """
    A greedy approach to find stable server positions for various server types inside one rack
    """
    global global_rack_signature
    
    #tohash = dict()
    #for key,value in enumerate(servers_to_place):
    #    tohash[str(key)+str(value)] = 1
    #signature = frozenset(tohash)
    servers_to_place = dict(servers_to_place_tuple)
    
    #signature = tuple(sorted(servers_to_place.items()))
    #if signature in global_rack_signature:
    #    return global_rack_signature[signature]
    
    all_servers = []
    for server_type, count in servers_to_place.items():
        specs = colors_info.get(server_type)
        if not specs:
            if 'LAN' in server_type:
                lan , sw_model = server_type.split('__')
                
                specs = [x for x in LANs[lan.split('_spine')[0]]['switch'] if x['model'] == sw_model][0]
                
            else:
                print(f"Warning: Specifications not found for server type '{server_type}'. Skipping.")
                continue
           
                
        specs['type'] = server_type
        
        for _ in range(count):
                all_servers.append(specs) # DEBUG_TAG: distribution_loop # append C
    
    if not all_servers:
        return {}, {}, {}  # Return empty dict for empty Counter
    all_servers = sort_device_before_placement(all_servers)
    physical_positions = dict()
    #physical_positions[43] = 'LAN_limit'
    #physical_positions[0] = 'occupied'
    for i in range(1,rack_height_u+1):
        physical_positions[i] = '0'
    pointer = rack_height_u + 1
    unstable_placement = {}
    rack_mapping = set() # slots that are occupied
    rack_mapping.add(pointer)
    min_stretch = float('inf')
    list_height_weight = list()
    for dev in all_servers:
        if dev['type'].startswith('LAN'):
                list_height_weight.append((dev['height'],dev['weight']))
                lan_sw = dev['type']
                if '_spine' not in dev['type']:
                    min_stretch = min(stretches[lan_sw],min_stretch)
                is_free = 0
                while not is_free and pointer:
                    pointer -=1
                    is_free = 1
                    for i in range(pointer , pointer-dev['height'], -1):
                        if  i in rack_mapping:
                            is_free = 0
                            break
                            #333333333
                    if is_free:
                        pointer = pointer - dev['height'] +1
                        unstable_placement[lan_sw+'_'+str(pointer)] = pointer
                        
                        for i in range(pointer,pointer+dev['height'] ):
                            physical_positions[i] = lan_sw+'_'+str(pointer)
                            rack_mapping.add(i)
                        break
    max_limit = pointer -1
    lans_height_weight = list_height_weight.copy()
    cur_location = max(max_limit - min_stretch + 1,1)
    min_limit = 1
    counter = 0
    for dev in all_servers:
        if dev['type'].startswith('LAN'):
            continue
        counter += 1
        fail_flag = 1
        while fail_flag:
            
            if dev['height'] + cur_location < max_limit:
                list_height_weight.append((dev['height']+cur_location-1,dev['weight']))
                if is_rack_stable(list_height_weight):
                    for i in range(cur_location,cur_location+dev['height']):
                        physical_positions[i] = dev['type']+'_'+str(counter)
                    #unstable_placement[dev['type']+'_'+str(cur_location)] = cur_location
                    cur_location = cur_location + dev['height']
                    fail_flag = 0
            if fail_flag:
                physical_positions = {
                    k - 1 if not v.startswith('LAN') and not v.startswith('0') else k: v 
                    for k, v in physical_positions.items()
                }
                cur_location -= 1
    passed_dev = set()
    for pos,device in  physical_positions.items():
        if device != '0':
            if device not in passed_dev:
                passed_dev.add(device)
                unstable_placement[device] = pos
    


    final_placement =  unstable_placement.copy()
    
                
                


    #444444444444444  we will place all on top then move the heaviest to the bottome keeping the smallest cable stretch
       
    
    devices, lans, rackswitches, cables = calculate_cable_lengths(final_placement)
    
    best_placement = (final_placement, devices,lans, rackswitches, cables)
    return best_placement
    


In [261]:
def force_stable_add_device(device_to_add, rack_info,idd):
    if 'LAN' in device_to_add:
        lan = device_to_add.split('__')[0]
        device_metadata = LANs[lan]['switch'][0]
    else:
        device_metadata = colors_info[device_to_add]
    rack_mapping = set()
    for stable_dev_p,value in rack_info['stable_placement'].items():
        stable_dev = '_'.join(stable_dev_p.split('_')[:-1])
        if stable_dev_p.startswith('LAN'):
            device_height = LANs[stable_dev.split('__')[0]]['switch'][0]['height']
        else:
            device_height = colors_info[stable_dev]['height']
        for occu in range(value, value+device_height):
            rack_mapping.add(occu)
    device_loc = set()
    for free in range(rack_height_u - device_metadata['height'],0, -1):
        for occu in range(free,free+device_metadata['height']):
            device_loc.add(occu)
        if device_loc - rack_mapping == device_loc:
            rack_info['stable_placement'][device_to_add+'_'+str(idd)] = free
    print('sssss',device_to_add, rack_info)
    print(rack_mapping)
    print('bbb',rack_info['stable_placement'])
    return rack_info
        
                
    

In [262]:
def force_add_device(device_to_add,rack_info):
    #add to rack_config the device with newid qty:1
    for device in device_to_add:
        idd = len(rack_info['rack_config'])
        rack_info['rack_config'][device] = 1
        #add to stable_config the device with newid qty:1  try to be on top
        rack_info = force_stable_add_device(device, rack_info,idd)
    rack_info['device_info'], rack_info['lan_info'], rack_info['switches'] = calculate_cable_lengths(rack_info['stable_placement'])
    
    return rack_info

In [263]:

def get_rack_layouts(distributions, sw_scenario_divisor):
    
    distributions_info = dict()
    for i, rack_config in distributions.items():
        if rack_config:
            switches = dict()
            # Top-biased placement
            rack_config_tuple = tuple(sorted(rack_config.items()))
            stable_placement , devices, lans, rackswitches = find_stable_positions_greedy_complex(
                            rack_config_tuple, prioritize_top=True
            )
            distributions_info[i] = dict({'rack_config':rack_config.copy(), 'stable_placement': stable_placement.copy() ,'lan_info':copy.deepcopy(lans),
                                              'device_info':copy.deepcopy(devices), 'switches':copy.deepcopy(rackswitches)})
    #return  distributions_info

    return  distributions_info 
    added_device = set()   
    new_racking = dict()
    dist_keys =distributions_info.keys()
    i = -1
    is_dist_changed = 0
    nearest_rack = 0
    for i,dist in distributions_info.items():
        
        if not i % sw_scenario_divisor:
            nearest_rack = i
            continue
        
        #lans_i = set(['_'.join(x.split('_')[:-1]) for x in distributions_info[i]['lan_info'] if x.startswith('LAN')])
        
        devs = set([x for x in distributions_info[i]['stable_placement'] if not x.startswith('LAN')])
        for dev in devs:
            device = '_'.join(dev.split('_')[:-1])
            not_found_lans = set([x+'__'+LANs[x]['switch'][0]['model'] for x in LANs if colors_info[device][x]['count'] > 0])
            for lan_sw in not_found_lans:
                lan_filled = 0
                ii = nearest_rack
                lanname = lan_sw.split('__')[0]
                lan = [ x for x in distributions_info[ii]['lan_info'] if x.startswith(lan_sw) ][0]
                added_device.add(lan_sw)
                lan_filled = 1
                lan_pos = distributions_info[ii]['lan_info'][lan]['position']
                rack_diff = abs(ii - i)
                absolute_sw_to_dev = rack_height_u - lan_pos + (2*racktop_to_ceiling) + (rack_diff*rack_to_rack) +(2*device_to_rackside) + rack_height_u - distributions_info[i]['stable_placement'][dev]
                absolute_sw_to_dev = absolute_sw_to_dev*unit_to_cm/100
                no_ports = colors_info[device][lanname]['count']
                speed_ratio =  colors_info[device][lanname]['speed'] / LANs[lanname]['switch'][0]['speed']
                device_position = distributions_info[i]['stable_placement'][dev]
                # absolute cable lengeth in meters:                   
                cable = get_the_right_cable(LANs[lanname]['switch'][0]['model'],absolute_sw_to_dev,colors_info[device][lanname]['speed'])
                no_of_cables = colors_info[device][lanname]['count'] * speed_ratio
                distributions_info[i]['device_info'][dev][lan] = dict()
                distributions_info[i]['device_info'][dev][lan]['cable_count'] = colors_info[device][lanname]['count']
                distributions_info[i]['device_info'][dev][lan]['speed'] = colors_info[device][lanname]['speed']
                distributions_info[i]['device_info'][dev][lan]['to_external_rack'] = ii
                if speed_ratio < 1:
                    distributions_info[ii]['lan_info'][lan]['half_for_split'] +=1
                    distributions_info[i]['device_info'][dev][lan]['half_for_split'] = 1
                    cable_to_ceil = cable['model']
                else:
                    distributions_info[ii]['lan_info'][lan]['full_no_split'] += 1
                    distributions_info[i]['device_info'][dev][lan]['full_no_split'] = 1
                distributions_info[i]['device_info'][dev][lan]['cables_count'] = no_of_cables
                distributions_info[i]['device_info'][dev][lan]['cable_type'] = cable
                if cable['model'] not in distributions_info[ii]['lan_info'][lan]['cables']:
                    distributions_info[ii]['lan_info'][lan]['cables'][cable['model']] = 0                    
                distributions_info[ii]['lan_info'][lan]['cables'][cable['model']] += no_of_cables
                distributions_info[ii]['lan_info'][lan]['minimum_cable_total_length'] += absolute_sw_to_dev * no_of_cables
                distributions_info[ii]['lan_info'][lan]['actual_cable_total_length'] += cable['length'] * no_of_cables

           
   
                    
                    
     
    

            
    return  distributions_info

In [264]:
i = 5
[i+d for x in range(1,1000 ) for d in (x, -x) if (d >= 0 or i+d >= 0) and i+d <= 15]

[6, 4, 7, 3, 8, 2, 9, 1, 10, 0, 11, 12, 13, 14, 15]

In [265]:
def otpimize_rack_layout(current_placement, device_info):

    """
    Place devices in the rack considering cable constraints and existing placements.
    
    Args:
        current_placement: Dictionary of device_id to position in rack
        device_info: Dictionary containing device connection information
        Cables: Global dictionary of cable specifications
        colors_info: Global dictionary of device specifications including height
        
    Returns:
        Tuple of (updated_current_placement, updated_device_info) with new positions
    """
    # Create copies of the input dictionaries to avoid modifying originals
    updated_placement = current_placement.copy()
    updated_device_info = {k: v.copy() for k, v in device_info.items()}
    
    # Create a set of occupied positions for quick lookup
    occupied_positions = set()
    for device, pos in updated_placement.items():
        # Get device type (before the first underscore or number)
        device_type = '_'.join(device.rsplit('_', 1)[:-1])
        if device_type in colors_info:
            height = colors_info[device_type]['height']
           
        else: 
            if 'spine' in device_type:
                height =  Spines[device_type.split('__')[0]]['switch'][0]['height']
            else:
                height =  LANs[device_type.split('__')[0]]['switch'][0]['height']
         # Mark all positions from pos to pos+height-1 as occupied
        occupied_positions.update(range(pos, pos + height))
    
    # Sort devices by their current position (lowest first) to prioritize moving them lower
    sorted_devices = sorted(
        [dev for dev in updated_device_info.items() if not dev[0].startswith('LAN_')],
        key=lambda x: x[1]['position']
    )
    
    for device_id, device_data in sorted_devices:
        current_pos = device_data['position']
        device_type = '_'.join(device.rsplit('_', 1)[:-1])
       
        if device_type not in colors_info:
            continue  # Skip unknown device types
            
        height = colors_info[device_type]['height']
        
        # Find all connected LAN devices and their constraints
        lan_constraints = []
        for lan_device, connection in device_data.items():
            if isinstance(connection, dict) and 'cable_type' in connection and lan_device in updated_placement:
                cable_model = connection['cable_type']['model']
                in_rack_stretch = connection['cable_type']['in_rack_stretch']
                lan_pos = updated_placement[lan_device]
                lan_device_type = lan_device.split('_')[0]
                lan_height = colors_info.get(lan_device_type, {}).get('height', 1)
                
                lan_constraints.append({
                    'lan_pos': lan_pos,
                    'lan_height': lan_height,
                    'max_distance': in_rack_stretch,
                    'cable_model': cable_model,
                    'lan_device': lan_device
                })
        
        # Try to find the best position for this device (try to go as low as possible)
        best_pos = None
        
        # Determine the lowest possible position we can try (considering height)
        min_possible_pos = 1
        max_possible_pos = current_pos  # We're only trying to move downward
        
        # Search from lowest possible position up to current position
        for pos in range(min_possible_pos, max_possible_pos + 1):
            # Check if this position and height would fit
            
            if any(p in occupied_positions for p in range(pos, pos + height)):
                continue
            
            # Check all LAN constraints
            valid = True
            for constraint in lan_constraints:
                distance = abs(pos - constraint['lan_pos'])
                if distance > constraint['max_distance']:
                    valid = False
                    break
            
            if valid:
                best_pos = pos
                break
                
        
        # If we found a valid position (lower than current), update everything
        if best_pos is not None and best_pos < current_pos:
            # Remove old position from occupied positions
            for i in range(height):
                if current_pos + i in occupied_positions:
                    occupied_positions.remove(current_pos + i)
            
            # Update the position in both dictionaries
            updated_placement[device_id] = best_pos
            updated_device_info[device_id]['position'] = best_pos
            
            # Add new position to occupied positions
            for i in range(height):
                occupied_positions.add(best_pos + i)
    
    return updated_placement, updated_device_info
    
    

In [266]:
def arrange_racks_best_(distributions_info):
    rack_updates = {}
    for i in distributions_info:
        if 'rack_config' in distributions_info[i]:
            rack_updates[i] = {}
            #xx = {'compute_nodes_1': 29, 'compute_nodes_2': 27, 'compute_nodes_3': 25, 'LAN_2__Z_9xxx_4': 21, 'LAN_3__S_xx64_5': 19, 'compute_nodes_6': 17, 'compute_nodes_7': 15, 'compute_nodes_8': 13}    
            rack_updates[i]['palcement'], rack_updates[i]['laninfo'] = otpimize_rack_layout( distributions_info[i]['stable_placement'],distributions_info[i]['device_info'])
            #otpimize_rack_layout( xx,distributions_info[i]['device_info'])

    for i in rack_updates:
       distributions_info[i]['bottom_place_periority'], distributions_info[i]['bottom_laninfo_periority'] = rack_updates[i]['palcement'], rack_updates[i]['laninfo'] 
    return distributions_info

In [267]:
number = 1234567890
print(f"Formatted: {number:,}") 

Formatted: 1,234,567,890


In [268]:
import numpy as np
import os, pickle
import time
import multiprocessing as mp
from collections import Counter
from itertools import product, permutations
from math import ceil, floor
import logging
import random
import functools, itertools
#from collections import deque
from copy import deepcopy

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def calculate_box_wattage(box,color_metadata):
    total = 0
    color_metadata = dict(color_metadata_tuple)
    box = dict(box_tuple)
    for color, count in box.items():
        meta = color_metadata[color]
        total += count * meta[1]
    return total


def calculate_box_height(box, color_metadata):
    total = 0
    color_metadata = dict(color_metadata_tuple)
    box = dict(box_tuple)
    for color, count in box.items():
        meta = color_metadata[color]
        total += count * meta[2]
    return total

@functools.lru_cache(maxsize=1000)
def calculate_box_wattage_height(box_tuple, color_metadata_tuple):
    total_wattage= 0
    total_height = 0
    color_metadata = dict(color_metadata_tuple)
    box = dict(box_tuple)
    
    for color, count in box.items():
        meta = color_metadata[color]
        total_wattage += count * meta[1]
        total_height += count * meta[2]
    return total_wattage, total_height
    
@functools.lru_cache(maxsize=1000)
def check_box_limits(box_tuple, color_metadata_tuple):
    color_metadata = dict(color_metadata_tuple)
    box = dict(box_tuple)
    wattage = height = 0
    for color, count in box.items():
        meta = color_metadata[color]
        wattage += count * meta[1]
        height += count * meta[2]
        # Early exit if limits exceeded
        if wattage > max_box_wattage or height > max_box_height:
            return False, wattage, height
    return True, wattage, height

def get_total_cable_lengths(distributions_info):
    global_minimum_length = 0
    global_actual_length = 0
    global_switches = dict()
    cables = dict()

    for rack_info in distributions_info:
        if 'lan_info' in distributions_info[rack_info]:
            for lan in distributions_info[rack_info]['lan_info']:
                global_minimum_length += distributions_info[rack_info]['lan_info'][lan]['minimum_cable_total_length']
                global_actual_length += distributions_info[rack_info]['lan_info'][lan]['actual_cable_total_length']
                for cable in distributions_info[rack_info]['lan_info'][lan]['cables']:
                    if cable not in cables:
                        cables[cable] = 0
                    cables[cable] += distributions_info[rack_info]['lan_info'][lan]['cables'][cable]
        if 'switches' in distributions_info[rack_info]:
            for sw_lan in distributions_info[rack_info]['switches']:
                if sw_lan not in global_switches:
                    global_switches[sw_lan] = 0
                global_switches[sw_lan] += distributions_info[rack_info]['switches'][sw_lan]
    
    distributions_info['cables'] = {'global_minimum_length':global_minimum_length, 'global_actual_length':global_actual_length,
                                   'cables':cables }
    distributions_info['switches'] = global_switches

    return distributions_info

def display_distribution_filled_only(distributions_info, color_metadata_tuple, total_balls, balls_placed_overall):
    global best_count
    print(f"\nValid Distribution Found (Iterative Greedy - Filled Racks Only):")
    current_total_balls = sum(balls_placed_overall.values())
    total_wattage = 0
    total_height = 0
    total_count = 0
    filled_boxes = []
    # Create a dictionary to store unique rack configurations
    unique_racks = {}
    rack_summary = []
    
    # First pass: Identify and group identical racks
    for i, info in enumerate(distributions_info):
        if 'rack_config' in distributions_info[info]:
            # Create a hashable representation of the rack configuration
            rack_config = distributions_info[info]['rack_config']
            # Convert the rack_config dictionary to a tuple of sorted items for hashability
            rack_tuple = tuple(sorted(rack_config.items()))

            # Store the rack details with its ID
            rack_details = {
                'rack_id': i+1,
                'config': rack_config,
                'stable_placement': distributions_info[info].get('stable_placement', []),
                'bottom_place_periority': distributions_info[info].get('bottom_place_periority', []),
                'lan_info': distributions_info[info].get('lan_info', {}),
                'device_info': distributions_info[info].get('device_info', {})
            }

            # If we've seen this configuration before, append to the list
            if rack_tuple in unique_racks:
                unique_racks[rack_tuple]['rack_ids'].append(i+1) # append D
                unique_racks[rack_tuple]['count'] += 1
            else:
                # First time seeing this configuration
                unique_racks[rack_tuple] = {
                    'rack_ids': [i+1],
                    'count': 1,
                    'details': rack_details
                }

        # Track the total count regardless of whether we're processing racks
        if 'stable_placement' in distributions_info[info]:
            total_count += len(distributions_info[info]['stable_placement'])

    # Second pass: Generate summary output
    filled_boxes = []
    rack_counts = 0
    for rack_tuple, data in unique_racks.items():
        rack_config = data['details']['config']
        rack_config_tuple = tuple(sorted(rack_config.items()))
        box_contents = ", ".join(f"{count} {color}" for color, count in rack_config.items())
        box_wattage, box_height = calculate_box_wattage_height(rack_config_tuple, color_metadata_tuple)
        total_wattage += box_wattage * data['count']  # Multiply by number of identical racks
        total_height += box_height * data['count']  # Multiply by number of identical racks
        ball_count = sum(rack_config.values())

        # Format rack IDs nicely
        if len(data['rack_ids']) > 1:
            rack_id_str = f"Racks {', '.join(map(str, data['rack_ids']))}"
        else:
            rack_id_str = f"Rack {data['rack_ids'][0]}"

        # Add number of identical racks if more than one
        count_str = f" ({data['count']} identical racks)" if data['count'] > 1 else ""
        rack_counts += data['count']

        filled_boxes.append(f"\x1b[1m -------------------------{rack_id_str} info: {count_str}------------------------------\x1b[0m")
        filled_boxes.append(f"\x1b[1;4m-->{rack_id_str}:\x1b[0m \n {box_contents} (Wattage: {box_wattage}, Height: {box_height}, count:{ball_count})")
        filled_boxes.append(f"-->\x1b[1;4mdevice placement: in the rack \x1b[0m \n {data['details']['stable_placement']}")
        filled_boxes.append(f"-->\x1b[1;4mbottom periority placement: in the rack \x1b[0m \n {data['details']['bottom_place_periority']}")
        filled_boxes.append(f"-->\x1b[1;4maggregated LAN info: \x1b[0m \n {data['details']['lan_info']}")
        #filled_boxes.append(f"-->\x1b[1;4mDevice detailed info: \x1b[0m \n {data['details']['device_info']}")   

 

    filled_boxes.append(f"\x1b[1;4m--------------------Total distribution info ({rack_counts}) racks, in {len(unique_racks)} groups------------------------------ \x1b[0m \n {distributions_info['cables']} \n {distributions_info['switches']}")
    if filled_boxes:
        for box_info in filled_boxes:
            print('rack_info', box_info)
        print(f"  Total Wattage: {total_wattage}, Total Height: {total_height}, Total Balls Distributed:{total_count} {current_total_balls}")
    else:
        print(f"  No racks were filled. Total Balls Distributed: {current_total_balls}")
    if current_total_balls != total_balls:
        print(f"  WARNING: Total balls distributed ({current_total_balls}) does not match the expected total ({total_balls})!")
    
    with open(f"Latest_Racks_{project}_{best_count}_{rack_counts}R_{sum(distributions_info['switches'].values())}S_{len(unique_racks)}g.pkl", 'wb') as f:
        pickle.dump([colors_info, LANs,Spines, Cables, Rack_rows, distributions_info], f)
    best_count += 1
@functools.lru_cache(maxsize=1000)
def is_switch_relevent(rack):
    alllans = set()
    allnodes = ""
    include_lan = set()
    execlude_lan = set()
    for device in rack:
        if 'LAN' in device:
            switch = device.split('__')[-1]
            lan = device.split('__')[0]
            alllans.add(lan)
    is_there_device = 0
    for device in rack:
        if 'LAN' not in device:
            for lan in alllans:
                if colors_info[device][lan]['count'] > 0:
                    include_lan.add(lan)
    execlude_lan = alllans - include_lan
    return list(execlude_lan)

def get_distribution_signature(distribution):
        #start_time = time.time()
        filled_boxes_signature = tuple(sorted(tuple(sorted(box.items())) for _,box in distribution.items() if box))
        #end_time = time.time()
        #print(f"Update: the signing the distribution is calcualted in {end_time - start_time:.6f} seconds")
        return filled_boxes_signature

def format_time_difference_compact(seconds):
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    days, hours = divmod(hours, 24)

    parts = []
    if days > 0:
        parts.append(f"{int(days)}d")
    if hours > 0:
        parts.append(f"{int(hours)}h")
    if minutes > 0:
        parts.append(f"{int(minutes)}m")
    parts.append(f"{seconds:.3f}s")  # Show seconds always, with 3 decimal places for compactness

    return " ".join(parts) or "0.000s" 
    
def find_valid_distributions_iterative_greedy_adaptive( initial_num_boxes,  total_balls):
    import cProfile

    profiler = cProfile.Profile()
    profiler.enable()
    colors = [x for x in colors_info.keys() if colors_info[x]['count'] > 0]
    endless_color = itertools.cycle(colors)
    #create a dictionary same like the color_info which include the LANs various switches
    switch_info = dict()
    clans = set()
    current_lans = []
    for color in colors:
        for lan in colors_info[color]:
            if lan.startswith('LAN') and colors_info[color][lan]['count'] > 0:
                clans.add(lan)
    current_lans = list(clans)
    for lan in current_lans:
        switch_info[lan] = copy.deepcopy(LANs[lan]['switch'])
    colors_wattage = {color: colors_info[color]['wattage'] for color in colors}
    colors_wattage.update({lan+'__'+switch_info[lan][0]['model']: switch_info[lan][0]['wattage'] for lan in list(switch_info.keys())})
    colors_wattage.update({lan+'_spines'+'__'+switch_info[lan][0]['model']: switch_info[lan][0]['wattage'] for lan in list(switch_info.keys())})
    
    colors_height = {color: colors_info[color]['height'] for color in colors}
    colors_height.update({lan+'__'+switch_info[lan][0]['model']: switch_info[lan][0]['height'] for lan in list(switch_info.keys())})
    colors_height.update({lan+'_spines'+'__'+switch_info[lan][0]['model']: switch_info[lan][0]['height'] for lan in list(switch_info.keys())})
    
    colors_weight = {color: colors_info[color]['weight'] for color in colors}
    colors_weight.update({lan+'__'+switch_info[lan][0]['model']: switch_info[lan][0]['weight'] for lan in list(switch_info.keys())})
    colors_weight.update({lan+'_spines'+'__'+switch_info[lan][0]['model']: switch_info[lan][0]['weight'] for lan in list(switch_info.keys())})

    color_metadata = {
    color: (
        color.split('__')[-1],  # Gets the part after last '__',
        colors_wattage[color],
        colors_height[color],
        colors_weight[color],
        )
        for color in set(colors_wattage.keys()).union(colors_height.keys())
    }
    color_metadata_tuple = tuple(sorted(color_metadata.items()))
    
    max_possible_boxes = total_balls
    valid_distributions = dict()
    valid_counter = 1
    seen_distributions = set()

    

    num_boxes_to_try = list(range(1, initial_num_boxes + max_possible_boxes + 1))
    #random.shuffle(num_boxes_to_try)
    lan_len = len(LANs)
    switch_scenarios = []
    #switch_scenarios.append("switch_per_1_rack")
    #switch_scenarios.append("switch_per_2_Racks")
    switch_scenarios.append("switch_per_3_Racks")
    #switch_scenarios.append("switch_per_4_Racks")
    #switch_scenarios.append("switch_per_5_Racks")
    #switch_scenarios.append("switch_per_6_Racks")
    #switch_scenarios.append("switch_per_7_Racks")
    #switch_scenarios.append("switch_per_8_Racks")
    swtich_index = 0
    start_time = time.time()
    current_switches = copy.deepcopy(switch_info)
    swcounter = 0 
    iter_counter = 1
    switch_to_add = dict()
    spine_to_add = dict()
    for lan in switch_info:
        switch_to_add[lan] = (lan+ '__' + switch_info[lan][0]['model'],LANs[lan]['fixedqty'])
        spine_to_add[lan] = (lan+'_spines'+ '__' + switch_info[lan][0]['model'],Spines[lan+'_spines']['fixedqty'])
    best_minimum_length = float('inf')
    best_actual_length = float('inf')
    best_distributions = []
    best_min_devices = float('inf')
    possible_colors = [c for c in colors]
    rng = random.Random(43)
    best_LAN_check = []
    for _ in LANs:
        best_LAN_check.append(float('inf'))
    sc_counter = 0
    
    while iter_counter:
        
        sc_counter += 1
        distributions_done = 0
        
        #end_time = time.time()
        #print(f"Update: new iter_counter No {len(seen_distributions)} is calcualted in {end_time - start_time:.6f} sec after passing {iter_counter} iterations and rack cash {len(global_rack_signature)}",end='\r', flush=True)
        #start_time = time.time()
        distributions = dict()
        endless_switch_scenarios = itertools.cycle(switch_scenarios)
        for sw_scenario in endless_switch_scenarios:
            if distributions_done:
                break
            distributions = dict()
            distributions[0] = Counter() 
            #if sw_scenario.split('_')[2] == '1':
            #    for box in distributions:
            #        for sw in switch_to_add:
            #            box[sw] = 1

            box_index = 0       
            #else:
            #    break
            sw_scenario_divisor = int(sw_scenario.split('_')[2])
            stub_rack = distributions[box_index]
            balls_placed = {color: 0 for color in colors}
            #lan_prefixes_to_check = [lan for lan in main_color_info if 'LAN' in lan]
            used_lans = set()
            for lan in switch_to_add:
                added = 0
                if not box_index % sw_scenario_divisor:
                    switch_toadd = switch_to_add.get(lan) # Direct dict access
                    if switch_toadd:
                        if switch_toadd[1] > 0:
                            switch_key = switch_toadd[0]
                            distributions[box_index][switch_key] = 1
                            switch_to_add[lan] = (switch_key,switch_to_add[lan][1] - 1)
                            used_lans.add(lan)
                            added = 1
                    if added:
                        switch_toadd = spine_to_add.get(lan) # Direct dict access
                        if switch_toadd:
                            if switch_toadd[1] > 0:
                                switch_key = switch_toadd[0]
                                distributions[box_index][switch_key] = 1
                                spine_to_add[lan] = (switch_key,spine_to_add[lan][1] - 1)
                               
                
                        
                    
                    
            temp_box = distributions[box_index].copy()
            
            

            remaining_balls = {c: colors_info[c]['count'] - balls_placed[c] for c in colors if colors_info[c]['count'] - balls_placed[c]> 0}
            
            possible_colors = list(remaining_balls.keys())
            possible_len = len(possible_colors)
            temp_balls_placed = balls_placed.copy()
            temp_remaining_balls = remaining_balls.copy() 
            
            if not possible_colors:
                end_time = time.time()
                print(f"Update: failed combination {len(seen_distributions):,} is calcualted in {end_time - start_time:.6f} sec after passing {iter_counter:,} iterations and rack cash {len(global_rack_signature)}",end='\r', flush=True)
                start_time = time.time()
                break
            if sc_counter == 100:
                sc_counter = 0
                for i in range(possible_len-1, 0, -1):
                    j = rng.randint(0, i)
                    possible_colors[i], possible_colors[j] = possible_colors[j], possible_colors[i]

            scenarios = list()

            for iddc in range(101-sc_counter,0,-1):

                #scenarios.append(("single_color", color))
                for color in possible_colors:
                    scenarios.append(("partial_color_"+str(iddc), color)) # append E
                #scenarios.append(("mixed_fill_initial", color))
            if not sc_counter % 10:
                sc_len = len(scenarios)  - 1
                for i in range(sc_len-1, 0, -1):
                    j = rng.randint(0, i)
                    scenarios[i], scenarios[j] = scenarios[j], scenarios[i]


            #random.shuffle(scenarios) # Try scenarios in a random order for each box


            applied_scenario = False
            add_amount = 1
            add_flag = 1
            endless_scenarios = itertools.cycle(scenarios)
            
            for main_color_1 in endless_color:
                if distributions_done:
                    break
                for scenario_type, main_color_2 in endless_scenarios:
                    if iter_counter < (0.1*max_iter) or iter_counter > (0.5*max_iter):
                        main_color = main_color_1
                    else:
                        main_color = main_color_2 
                    
                    if  not add_flag:
                        add_flag = 1   
                        color_lans = set()
                        for item in temp_box:
                            if not item.startswith('LAN'):
                                for key in colors_info[item]:
                                    if key.startswith('LAN'):
                                        if colors_info[item][key]['count'] > 0:
                                            color_lans.add(key)
                        unused_lans = used_lans - color_lans
                        used_lans = set()
                        if unused_lans:
                           
                            for item in unused_lans:
                                temp_box = {k: v for k, v in temp_box.items() if not k.startswith(item)}
                                
                        else:
                            used_lans = set()
                            distributions[box_index] = temp_box.copy()
                            #print('tempppp',temp_box)
                            #print('dddddd',box_index,distributions[box_index])


                            balls_placed.update(temp_balls_placed)
                            remaining_balls.update(temp_remaining_balls)
                            applied_scenario = True

                            if sum(balls_placed.values()) == total_balls:
                                distributions_done = 1
                                break
                            box_index += 1
                            distributions[box_index] = Counter()
                            used_lans = set()
                            for lan in switch_to_add:
                                added = 0
                                if not box_index % sw_scenario_divisor :
                                    switch_toadd = switch_to_add.get(lan) # Direct dict access
                                    if switch_toadd:
                                        if switch_toadd[1] > 0:
                                            switch_key = switch_toadd[0]
                                            distributions[box_index][switch_key] = 1
                                            switch_to_add[lan] = (switch_key,switch_to_add[lan][1] - 1) 
                                            used_lans.add(lan)
                                            added = 1                                    
                                    if added:
                                        switch_toadd = spine_to_add.get(lan) # Direct dict access
                                        if switch_toadd:
                                            if switch_toadd[1] > 0:
                                                switch_key = switch_toadd[0]
                                                distributions[box_index][switch_key] = 1
                                                spine_to_add[lan] = (switch_key,spine_to_add[lan][1] - 1)



                            temp_box = distributions[box_index].copy()
                    if not remaining_balls.get(main_color):
                        break

                    add_flag = 0
                    main_color_info = colors_info[main_color]  # Cache dict lookup

                    percentage = float(int(scenario_type.split('_')[2])/100)
                    can_add = True
                    temp_box_keys = set(temp_box.keys())
                    temp_box_tuple = tuple(sorted(temp_box.items()))

                    box_limits, box_wattage, box_height = check_box_limits(temp_box_tuple, color_metadata_tuple)
                    add_amount = min(ceil(main_color_info['count'] * percentage), temp_remaining_balls[main_color],
                                    floor((max_box_wattage-box_wattage)/colors_wattage[main_color]),
                                    floor((max_box_height-box_height)/colors_height[main_color]))


                    if add_amount > 0:
                        box_wattage += add_amount * colors_wattage[main_color]
                        box_height += add_amount * colors_height[main_color]
                        temp_box[main_color] = temp_box.get(main_color, 0) + add_amount
                        temp_box_tuple = tuple(sorted(temp_box.items()))
                        temp_balls_placed[main_color] += add_amount
                        temp_remaining_balls[main_color] -= add_amount
                        add_flag = 1
                    iteratecolor = 1
                    removed_sw = []
                    cycled = 1
                    twice_state = 1
                    ball_added = 1

                    twice_state -=1
                    others = [x for x in temp_remaining_balls.keys() if x != main_color and temp_remaining_balls[x] > 0]
                    if temp_remaining_balls.get(main_color, 0) > 0: 
                        others = itertools.chain(others, [main_color])

                    recent_sw = []
                    for other_color in others:
                        if not temp_remaining_balls.get(other_color):
                            continue

                        temp_box_keys = set(temp_box.keys())
                        other_color_info = colors_info[other_color]  # Cache dict lookup
                        other_lan_prefixes_to_check = [lan for lan in other_color_info if 'LAN' in lan]

                        for lan in other_lan_prefixes_to_check:
                            # Check count first (cheaper than string operations)
                            if int(other_color_info[lan]['count']) <= 0:
                                continue
                        add_amount = min(ceil(other_color_info['count'] * percentage), temp_remaining_balls[other_color],
                                    floor((max_box_wattage-box_wattage)/colors_wattage[other_color]),
                                    ceil((max_box_height-box_height)/colors_height[other_color]))

                        if add_amount > 0:
                            box_wattage += add_amount * colors_wattage[other_color]
                            box_height += add_amount * colors_height[other_color]
                            temp_box[other_color] = temp_box.get(other_color, 0) + add_amount
                            temp_box_tuple = tuple(sorted(temp_box.items()))
                            temp_balls_placed[other_color] += add_amount
                            temp_remaining_balls[other_color] -= add_amount
                            add_flag = 1

                
                            

                
            if sum(balls_placed.values()) == total_balls:
                
                remove_indices = []
                for i,rack in distributions.items():
                    remove_rack = 1
                    for item in rack:
                        if 'LAN' not in str(item):
                            remove_rack = 0
                            break
                    if remove_rack:
                        remove_indices.append(i) # DEBUG_TAG: append A # append A
                for i in reversed(remove_indices):  
                    del distributions[i]
                switch_to_add = dict()
                for lan in switch_info:
                    switch_to_add[lan] = (lan+ '__' + switch_info[lan][0]['model'],LANs[lan]['fixedqty'])
                    spine_to_add[lan] = (lan+'_spines'+ '__' + switch_info[lan][0]['model'],Spines[lan+'_spines']['fixedqty'])
                #print('sssssss',switch_to_add)
                signature = get_distribution_signature(distributions)
                if signature not in seen_distributions:

                    seen_distributions.add(signature)
                    distributions_info = get_rack_layouts(distributions,sw_scenario_divisor)

                    distributions_info = get_total_cable_lengths(distributions_info)
                    total_devices = 0
                    for _, rack in distributions.items():
                            total_devices  += sum(rack.values())
                    LAN_check = [float('inf')] * len(LANs)
                    for id in range(len(LANs)):
                        for key in distributions_info['switches']:
                            if 'LAN_'+str(id) in key:
                                LAN_check[id] = distributions_info['switches'][key]
                                break

                    actual_length_check = distributions_info['cables']['global_actual_length']
                    total_length_check = sum(LAN_check)
                    if (actual_length_check < best_actual_length) or \
                    (actual_length_check == best_actual_length and LAN_check[0] < best_LAN_check[0]) or \
                     (actual_length_check == best_actual_length and LAN_check[0] == best_LAN_check[0] and total_devices < best_min_devices) :
                        best_min_devices = total_devices
                        best_LAN_check = LAN_check.copy()
                        best_distributions = distributions_info
                        best_actual_length = distributions_info['cables']['global_actual_length']
                        distributions_info = arrange_racks_best_(distributions_info)
                        display_distribution_filled_only(distributions_info, color_metadata_tuple, total_balls, balls_placed)
                        valid_distributions[valid_counter] = list(distributions) # DEBUG_TAG: valid_distributions #append J
                        valid_counter += 1
                    # Potentially break here if you only need one solution

                
                if iter_counter > max_iter:
                    profiler.disable()
                    profiler.print_stats(sort='cumtime')  # Sort by cumulative time
                    return seen_distributions, valid_distributions, iter_counter
                
                if iter_counter/500 == iter_counter //500:
                    #print(f"passing the {iter_counter} of {max_iter}")
                    end_time = time.time()
                    print(f"Update: the seen/validated distribution No {len(seen_distributions):,} is calcualted in {end_time - start_time:.6f} sec after passing {int(iter_counter):,} iterations and rack cash {len(global_rack_signature)}",end='\r', flush=True)
                    start_time = time.time()
                iter_counter += 1
                    #end_time = time.time()
                    #print(f"Update: the seen/validated distribution No{len(seen_distributions)} is calcualted in {end_time - start_time:.6f} seconds")
                    #start_time = time.time()
    profiler.disable()
    profiler.print_stats(sort='cumtime')  # Sort by cumulative time
    return seen_distributions, valid_distributions, iter_counter

In [269]:
def main_iterative_greedy_adaptive_with_height():
        
    
    num_devices = sum(info['count'] for info in colors_info.values())
    max_needed_wattage = sum(colors_info[color]['wattage'] * colors_info[color]['count'] for color in colors_info)
    # arrange the list of lANs that are found int eh color_info
    current_lans = []
    str_nodes= str(colors_info)
    for lan in LANs:
        if lan in str_nodes:
            current_lans.append(lan)
   
 
            
    #share this dictionary with the find_valid_distributions function
    initial_num_boxes = ceil(max_needed_wattage / max_box_wattage) + 2

    print(f"Distributing {num_devices} balls into initially {initial_num_boxes} boxes which consumes {max_needed_wattage} watts (Iterative Greedy Adaptive with Height):")

    start_time = time.time()
    

    seen_distributions, valid_distributions, iterations = find_valid_distributions_iterative_greedy_adaptive(
        initial_num_boxes, num_devices
    )

   
    
    
    end_time = time.time()
    elapsedtime = format_time_difference_compact(end_time-start_time)

    print(f"\nSummary: Found {len(seen_distributions)} unique distribution(s) with {len(valid_distributions)} times of optimum length after {iterations} iterations tries in {elapsedtime} seconds")


def global_main():
    global colors_info, LANs, Cables, max_box_wattage, max_box_height,  rack_height_u , rack_weight_kg, rack_width_mm, \
            rack_depth_mm, u_height_mm, rack_height_mm, rack_height_u, rack_cg_height_mm, unit_to_cm, u_height_mm, \
            device_to_rackside, racktop_to_ceiling, rack_to_rack, Rack_rows, stretches, best_count, project, max_iter,Spines
    
    colors_info = {
        'std_compute': {'count':330, 'wattage': 2380, 'height': 2, 'weight':28.7, 'LAN_1':{'count':1,'speed':400}, #IB
                                                                           'LAN_2':{'count':1,'speed':400},# FE storage
                                                                            'LAN_3':{'count':0, 'speed':400},#GPU_net
                                                                           'LAN_4':{'count':1,'speed':25}, # mamgt 1
                                                                           'LAN_5':{'count':1, 'speed':25}, #mgmt 2
                                                                           'LAN_6':{'count':0, 'speed':1}, #OOB
                                                                            
                      },
        'fat_compute': {'count':50, 'wattage': 2441, 'height': 2, 'weight': 28.7, 'LAN_1':{'count':1,'speed':400},
                                                                            'LAN_2':{'count':2,'speed':400},
                                                                            'LAN_3':{'count':0, 'speed':400},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':0, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                                                                            
                      
                      },
        'GPU_largeAI': {'count':5, 'wattage': 11632, 'height': 6, 'weight': 107, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':1,'speed':400},
                                                                            'LAN_3':{'count':8, 'speed':400},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                                                                            
                      
                      },
        'GPU_lowAI': {'count':10, 'wattage': 889, 'height': 2, 'weight': 27.5, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':1,'speed':400},
                                                                            'LAN_3':{'count':0, 'speed':400},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                      
                      },
        'FE_nodes': {'count':20, 'wattage': 586, 'height': 2, 'weight': 28.7, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':1,'speed':200},
                                                                            'LAN_3':{'count':0, 'speed':400},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                                                                           
                      
                      },
        'Mmgt': {'count':12, 'wattage': 527, 'height': 2, 'weight': 28.7, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':0,'speed':200},
                                                                            'LAN_3':{'count':0, 'speed':400},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                                                                           
                      
                      },
        'HPC_ScratchStorge': {'count': 20, 'wattage': 918, 'height': 2, 'weight':29.3, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':2,'speed':400},
                                                                            'LAN_3':{'count':0, 'speed':400},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                                                                         
                         },
        'HPC_Metadata': {'count':8, 'wattage': 814, 'height': 2, 'weight': 29.3, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':2,'speed':400},
                                                                            'LAN_3':{'count':0, 'speed':400},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                                                                            
                      
                      },
        'Net_mgmt': {'count': 2, 'wattage': 552, 'height': 2, 'weight':28.7, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':1,'speed':200},
                                                                            'LAN_3':{'count':0, 'speed':400},
                                                                            'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                                                                         
                          },
        'Powerscale': {'count': 1, 'wattage': 7000, 'height': 10, 'weight':200, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':16,'speed':100},
                                                                            'LAN_3':{'count':0, 'speed':400},
                                                                            'LAN_4':{'count':16, 'speed':25},
                                                                           'LAN_5':{'count':16, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                                                                    
                          },
        'ECS': {'count': 1, 'wattage': 10000, 'height': 28, 'weight':200, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':16,'speed':200},
                                                                            'LAN_3':{'count':0, 'speed':400},
                                                                            'LAN_4':{'count':16, 'speed':25},
                                                                           'LAN_5':{'count':16, 'speed':25},
                                                                           'LAN_6':{'count':0, 'speed':1},
                                                                    
                          },
                              
    }
    Rack_rows = {'Group':{'Racks':10,'group_count':2,'rack_to_rack':4.5,'Row_to_next_row':90}}
    
    LANs = { 'LAN_1':{'fixedqty':12, 'type':'400gbsNDR','topology': 'halfports_spine_leaf',
                  'switch':[{'model':'IB+400','ports':64,'speed':400, 'height':2,'wattage':2000, 'weight':20},
                         {'model':'IB_800','ports':64,'speed':800, 'height':2,'wattage':2000, 'weight':20},
                           ]},
              
              'LAN_2':{'fixedqty':14,'type':'400GbpsEth', 'topology': 'uplinks_spine_leaf',
'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':1304, 'uplink_count':4, 'uplink_speed':800,'weight':20},
                 {'model':'Z_6xxx','ports':64,'speed':200, 'height':1,'wattage':1304, 'uplink_count':4, 'uplink_speed':400,'weight':20},
        ]},
              'LAN_4':{'fixedqty':4,'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300, 'uplink_count':4, 'uplink_speed':100,'weight':12},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':4, 'uplink_speed':100,'weight':12},
                 ]},
              'LAN_5':{'fixedqty':4,'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':200,'uplink_count':4, 'uplink_speed':100,'weight':12},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':2, 'uplink_speed':100,'weight':12},
                 ]},
              'LAN_6':{'fixedqty':1000,'type':'1gbps','topology': 'uplinks_spine_leaf',
    'switch':[{'model':'SN_24','ports':24,'speed':1, 'height':1,'wattage':200, 'uplink_count':2, 'uplink_speed':25, 'weight':6},
             ]},
              'LAN_3':{'fixedqty':1,'type':'400GbpsEth','topology':'rails',
    'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
             ]},
           }
    Spines = { 'LAN_1_spines':{'fixedqty':8, 'type':'400gbsNDR','topology': 'halfports_spine_leaf',
                  'switch':[{'model':'IB+400','ports':64,'speed':400, 'height':2,'wattage':2000, 'weight':20},
                         {'model':'IB_800','ports':64,'speed':800, 'height':2,'wattage':2000, 'weight':20},
                           ]},
              
              'LAN_2_spines':{'fixedqty':8,'type':'400GbpsEth', 'topology': 'uplinks_spine_leaf',
'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':1304, 'uplink_count':4, 'uplink_speed':800,'weight':20},
                 {'model':'Z_6xxx','ports':64,'speed':200, 'height':1,'wattage':1304, 'uplink_count':4, 'uplink_speed':400,'weight':20},
        ]},
              'LAN_4_spines':{'fixedqty':2,'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300, 'uplink_count':4, 'uplink_speed':100,'weight':12},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':4, 'uplink_speed':100,'weight':12},
                 ]},
              'LAN_5_spines':{'fixedqty':2,'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':200,'uplink_count':4, 'uplink_speed':100,'weight':12},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':2, 'uplink_speed':100,'weight':12},
                 ]},
              'LAN_6_spines':{'fixedqty':1000,'type':'1gbps','topology': 'uplinks_spine_leaf',
    'switch':[{'model':'SN_24','ports':24,'speed':1, 'height':1,'wattage':200, 'uplink_count':2, 'uplink_speed':25, 'weight':6},
             ]},
              'LAN_3_spines':{'fixedqty':0,'type':'400GbpsEth','topology':'rails',
    'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
             ]},
           }
    Cables = {'IB+400':[{'model':'1.5m_400IB_copper','server_port_speed':400,'split':1,'length':1.5},
                                           {'model':'3m_400IB_copper','server_port_speed':400,'split':1,'length':3},
                                         {'model':'5m_400IB_fiber','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400IB_fiber','server_port_speed':400,'split':1,'length':7},
                                      { 'model':'10m_400IB_fiber','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400IB_fiber','server_port_speed':400,'split':1,'length':20},
                                      { 'model':'3m_400sIB_copper','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400sIB_fiber','server_port_speed':200,'split':2,'length':5},
                                      { 'model':'7m_400sIB_fiber','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400sIB_fiber','server_port_speed':200,'split':2,'length':10},
                                     {  'model':'20m_400sIB_fiber','server_port_speed':200,'split':2,'length':20},
                                     { 'model':'400IB_Morethan_20m_may_need_extra_hop','server_port_speed':400,'split':1,'length':200},
                                    { 'model':'200IB_Morethan_20m_may_need_extra_hop','server_port_speed':200,'split':2,'length':200},
                            ],
                            
          'IB_800':[{'model':'1.5m_800IB_copper','server_port_speed':800,'split':1,'length':1.5},
                                       {'model':'3m_800IB_copper','server_port_speed':800,'split':1,'length':3},
                                      { 'model':'8m_800IB_fiber','server_port_speed':800,'split':1,'length':5},
                                      { 'model':'7m_800IB_fiber','server_port_speed':800,'split':1,'length':7},
                                      { 'model':'10m_800IB_fiber','server_port_speed':800,'split':1,'length':10},
                                       {'model':'20m_800IB_fiber','server_port_speed':800,'split':1,'length':20},
                                      { 'model':'3m_800sIB_copper','server_port_speed':400,'split':2,'length':3},
                                     {  'model':'5m_800sIB_fiber','server_port_speed':400,'split':2,'length':5},
                                      { 'model':'7m_800sIB_fiber','server_port_speed':400,'split':2,'length':7},
                                      { 'model':'10m_800sIB_fiber','server_port_speed':400,'split':2,'length':10},
                                      { 'model':'20m_800sIB_fiber','server_port_speed':400,'split':2,'length':20},
                                     { 'model':'800IB_Morethan_20m_may_need_extra_hop','server_port_speed':800,'split':2,'length':200},
                                     { 'model':'400_Morethan_20m_may_need_extra_hop','server_port_speed':400,'split':2,'length':200},
                   ],
          'S_xx64':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                     {  'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},
                                    { 'model':'Morethan_20mEth_may_need_extra_hop','server_port_speed':25,'split':1,'length':200},
                   ],
          'S_xx32':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                       {'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},
                                       { 'model':'Morethan_20mEth_may_need_extra_hop','server_port_speed':25,'split':1,'length':200},
                   ],
          'SN_24':[{'model':'1.5m_1_copper','server_port_speed':1,'split':1,'length':1.5},
                                      { 'model':'3m_1_coppe','server_port_speed':1,'split':1,'length':3},
                                       {'model':'5m_1_copper','server_port_speed':1,'split':1,'length':5},
                                      { 'model':'7m_1_copper','server_port_speed':1,'split':1,'length':7},
                                      { 'model':'10m_1_copper','server_port_speed':1,'split':1,'length':10},
                                      { 'model':'20m_1_copper','server_port_speed':1,'split':1,'length':20}, 
                                      { 'model':'30m_1_copper','server_port_speed':1,'split':1,'length':30},
                                      {'model':'40m_1_copper','server_port_speed':1,'split':1,'length':40},
                                      { 'model':'Morethan_40m_may_need_batchpanel','server_port_speed':1,'split':1,'length':200},
                  ],
          'Z_9xxx':[{'model':'1.5m_400_copper_eth','server_port_speed':400,'split':1,'length':1.5},
                                       {'model':'3m_400_copper_eth','server_port_speed':400,'split':1,'length':3},
                                      { 'model':'5m_400_fiber_eth','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400_fiber_eth','server_port_speed':400,'split':1,'length':7},
                                     {  'model':'10m_400_fiber_eth','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400_fiber_eth','server_port_speed':400,'split':1,'length':20},
                                     {'model':'3m_100s_copper_eth','server_port_speed':100,'split':2,'length':3},
                                      { 'model':'5m_100s_fiber_eth','server_port_speed':100,'split':2,'length':5},
                                      { 'model':'7m_100s_fiber_eth','server_port_speed':100,'split':2,'length':7},
                                     {  'model':'10m_100s_fiber_eth','server_port_speed':100,'split':2,'length':10},
                                      { 'model':'20m_100s_fiber_eth','server_port_speed':100,'split':2,'length':20},
                                        {'model':'1.5m_400s_copper_eth','server_port_speed':200,'split':2,'length':1.5},
                                      { 'model':'3m_400s_copper_eth','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400s_fiber_eth','server_port_speed':200,'split':2,'length':5},
                                     {  'model':'7m_400s_fiber_eth','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400s_fiber_eth','server_port_speed':200,'split':2,'length':10},
                                      { 'model':'20m_400s_fiber_eth','server_port_speed':200,'split':2,'length':20},
                                    { 'model':'200_Morethan_20m_may_need_extra_hop','server_port_speed':200,'split':2,'length':200},
                                    { 'model':'400_Morethan_20m_may_need_extra_hop','server_port_speed':400,'split':1,'length':200},
                                    { 'model':'100_Morethan_20m_may_need_extra_hop','server_port_speed':100,'split':2,'length':200},
                   ],
         }         
            
    best_count = 0
    max_box_wattage = 17000
    max_box_height = rack_height_u = 42
    rack_weight_kg = 114.55
    rack_width_mm = 750
    rack_depth_mm = 1200
    u_height_mm = 44.45
    rack_height_mm = rack_height_u * u_height_mm
    rack_cg_height_mm = rack_height_mm / 2
    global_rack_signature = dict()
    unit_to_cm = u_height_mm /10
    # the following are in units
    device_to_rackside = 6.75
    racktop_to_ceiling = 4.5 # assumed 20cm
    rack_to_rack = 4.5 # assumed from side to the adjacent side no spacing
    # adding to the Cables the maximum stretch
    stretches = dict()
    for switch in Cables:
        for cable in Cables[switch]:
            cable['in_rack_stretch'] = floor((cable['length']*100/unit_to_cm) - (2* device_to_rackside))
    for lan in LANs:
        switch = LANs[lan]['switch'][0]['model']
        shortest_cable = min(Cables[switch], key=lambda x: x['length'])
        # Get its in_rack_stretch value
        stretches[lan+'__'+switch] = stretches[lan] = stretches[switch] = shortest_cable['in_rack_stretch']
    max_iter = 10000000
    project = 'MEWA'        
            
if __name__ == "__main__":
    global_main()
    main_iterative_greedy_adaptive_with_height()
    print('fffffffffffffffffffffffffffffffffffffff')

Distributing 459 balls into initially 63 boxes which consumes 1035520 watts (Iterative Greedy Adaptive with Height):


NameError: name 'cables' is not defined

In [ ]:
colors_info

In [ ]:
find_stable_positions_greedy_complex.cache_info()

In [ ]:
calculate_box_wattage_height.cache_info()

In [ ]:
is_switch_relevent.cache_info()

In [ ]:
ceil(5/3)

In [ ]:
xx = {'compute_nodes_1': 31, 'compute_nodes_2': 29, 'compute_nodes_3': 27, 'compute_nodes_4': 25, 'compute_nodes_5': 23, 'NVMenodes_6': 21, 'compute_nodes_7': 19, 'compute_nodes_8': 17, 'compute_nodes_9': 15, 'compute_nodes_10': 13, 'compute_nodes_11': 11, 'compute_nodes_12': 9}

In [ ]:
for x, v in xx:
    print(x, v)

In [ ]:
i = 4
ions = [i+d for x in range(-1,1000 ) for d in (x, -x) if i+d >= 0 and i+d != i and i+d<  len([0,1,2,3,4,5,6])]
ions